# Experiment analysis

Choose retained results → choose a recipe → review counts and evidence → preview → save a revision.
This is the main notebook for all supported analysis domains. The CLI uses the same backend.

No cell acquires data or changes a capture. Start with a catalogue row, or reopen a saved `selection.json`.
Imported non-queue folders also appear here after `analyse_results.py import --source PATH`.
See [the results guide](../docs/012_results_and_analysis.md#default-thesis-workflow) for measurement boundaries and command examples.


In [ ]:
from pathlib import Path
import json, os, sys
from IPython.display import display, Markdown, FileLink
import pandas as pd
candidates = [Path.cwd(), *Path.cwd().parents, Path('/workspace')]
ROOT = next((p for p in candidates if (p / 'scripts/framework/analysis_workspace.py').is_file()), None)
if ROOT is None:
    raise RuntimeError('Use ./scripts/dev.sh notebook from the framework repository')
sys.path.insert(0, str(ROOT / 'scripts'))
from framework import analysis_workspace as analysis

EXPORTS_ROOT = None  # Default: queue exports plus imported analysis exports.
catalogue = analysis.catalogue(EXPORTS_ROOT)
display(pd.DataFrame(catalogue))
display(pd.DataFrame(analysis.recipes()))
print('Catalogue verifies exported tables. Preview additionally verifies original evidence.')


## Choose results and the question

Set `CHOICE` to the catalogue row number, then set `RECIPE` using the list above. A saved selection takes precedence.
For domain recipes, `GROUP_BY` names the fields defining conditions (default: recorded case ID).
The baseline defaults to the first sorted condition and is frozen in the selection; review it below.
`TREATMENT` lists deliberately changed controls. Other unknown or conflicting controls prevent matched differences.
No independence assumption or control waiver is added automatically.

Use `COMPARISON` for an existing comparison dictionary, including explicit exclusions or not-applicable reasons.
Energy and packet recipes reuse recorded export annotations; missing attribution stays unavailable.
`ENERGY_SELECTION` or `NETWORK_SELECTION` can provide documented per-run attribution. Inspect run keys and controls below.

For a combined report, use `RECIPE = 'comparison'` and choose `DOMAINS`.
For multiple distinct campaigns or compact stack/follow-up relationships, use `EXPORTS` and `REQUESTS` instead of `CHOICE`.
Example request: `{'id':'authentication', 'kind':'tls_auth', 'filters':{'campaign_id':'YOUR_CAMPAIGN'}}`.
A `secure_boot_stack` request is linked with `stack_selection`; a `tls_auth_followup` request names its original `baseline_selection`.


In [ ]:
SPEC_FILE = os.environ.get('PQC_ANALYSIS_SPEC')
CHOICE = None  # An integer catalogue row, e.g. 0. Never automatically picks the latest export.
RECIPE = 'tls'
GROUP_BY = ['case_id']  # Examples: ['tls_kex'] or ['tls_auth', 'auth_mode'].
BASELINE = None  # Exact generated condition label; None records the first sorted condition.
TREATMENT = []  # Explicit changed fields, e.g. ['tls_kex', 'controls.tls.key_exchange_group'].
DOMAINS = ['timing', 'resources', 'tls']  # Used by RECIPE='comparison'.
COMPARISON = None  # Optional complete comparison dictionary, instead of GROUP_BY/BASELINE/TREATMENT.
NETWORK_SELECTION = None  # {run_key: {client_ip, server_ip, server_port, window, evidence_reference}}
ENERGY_SELECTION = None  # {'boundaries': {run_key: {electrical_boundary, evidence_reference}}}

EXPORTS = []  # Optional explicit archive names/paths for advanced multiple requests.
REQUESTS = []

if SPEC_FILE:
    path = Path(SPEC_FILE)
    spec = analysis.read_spec(path if path.is_absolute() else ROOT / path)
elif CHOICE is not None:
    if type(CHOICE) is not int or not 0 <= CHOICE < len(catalogue):
        raise ValueError('CHOICE must be a catalogue row number')
    chosen = catalogue[CHOICE]
    if chosen['status'] != 'outputs_verified':
        raise ValueError('The chosen export failed verification: ' + chosen['error'])
    filters = {'campaign_id': chosen['campaign']} if chosen['campaign'] is not None else {'_export': chosen['path']}
    details = analysis.describe([chosen['path']], filters)
    display(pd.DataFrame(details['controls']))
    display(pd.DataFrame(details['runs']))
    request = dict(id=RECIPE, kind=RECIPE, filters=filters)
    if RECIPE in analysis.comparison.RECIPES:
        request.update({'comparison': COMPARISON} if COMPARISON is not None else dict(group_by=GROUP_BY, baseline=BASELINE, treatment=TREATMENT))
        if RECIPE == 'comparison':
            request['domains'] = DOMAINS
        if NETWORK_SELECTION is not None:
            request['network'] = NETWORK_SELECTION
        if ENERGY_SELECTION is not None:
            request['energy'] = ENERGY_SELECTION
    spec = analysis.draft([chosen['path']], [request])
else:
    spec = analysis.draft(EXPORTS, REQUESTS, directory=EXPORTS_ROOT) if EXPORTS and REQUESTS else None

if spec is not None:
    display(pd.DataFrame(analysis.selection_audit(spec)))
    for selection in spec['selections']:
        if 'comparison' in selection:
            print(selection['id'], json.dumps(selection['comparison'], indent=2))
else:
    print('Set CHOICE or SPEC_FILE, then rerun from this cell.')


## Preview and presentation

The first preview shows a small primary set for each domain; every detailed frame is retained. Use its table IDs and metric names to refine subsequent previews.
Set `APPLY_PRESENTATION = True` to apply these options; a saved specification otherwise keeps its settings.
`FIGURE_METRICS = ['*']` plots available domain summaries; `[]` produces tables only.
`TABLE_IDS = []` keeps all tables. Labels and captions change presentation, not scientific values.
Every figure uses the full-precision statistic of its matching table. Missing observations stay missing.

For a detailed domain frame, add `(selection_id, frame_name)` to `DETAIL_FRAMES`; all detailed frames are saved as CSV regardless.
Tables may report a descriptive median while the matched-difference table explains why comparison is unavailable.


In [ ]:
APPLY_PRESENTATION = False
FIGURE_METRICS = None  # None keeps the recipe default; ['tls_handshake'] selects a metric; [] disables figures.
MEASUREMENTS = None  # None uses domain defaults; ['*'] shows all; ['tls_handshake', '*energy_j*'] selects metrics.
TABLE_IDS = []  # Optional table IDs shown in the preview index below.
LABELS = {}  # Original displayed condition -> thesis label.
ORDER = []  # Original displayed conditions in desired order.
CAPTIONS = {}  # Table ID -> thesis caption.
WIDTH_INCHES = 6.5
DETAIL_FRAMES = []  # Example: [('tls', 'tls_per_run')].

if spec is not None:
    if APPLY_PRESENTATION:
        options = dict(spec.get('presentation', {}), labels=LABELS, order=ORDER,
                       captions=CAPTIONS, tables=TABLE_IDS, width_inches=WIDTH_INCHES)
        if MEASUREMENTS is not None:
            options['measurements'] = MEASUREMENTS
        if FIGURE_METRICS is not None:
            options['figures'] = FIGURE_METRICS
        spec['presentation'] = options
    prepared = analysis.prepare(spec)
    for name, details in prepared['analyses'].items():
        print(name, 'available measurements:', details['available_measurements'])
    display(pd.DataFrame([{'table_id':t['id'], 'caption':t['caption'], 'rows':len(t['rows'])} for t in prepared['tables']]))
    display(pd.DataFrame(prepared['metrics'])[['table','metric','unit','scope']].drop_duplicates() if prepared['metrics'] else pd.DataFrame())
    analysis.display_preview(prepared)
    for selection_id, frame_name in DETAIL_FRAMES:
        data = prepared['analyses'][selection_id]['frames'][frame_name]
        display(Markdown('### ' + selection_id + ' / ' + frame_name))
        display(pd.DataFrame(data['rows'], columns=data['columns']))


## Save and reuse

Set `SAVE = True` to publish this selection. Each save creates a fresh revision, preserving earlier reports.
Outputs include CSV/LaTeX tables, SVG/PDF/PNG figures and their data, attempt audit, detailed domain CSVs,
method definitions, source identities, the pinned selection and checksums.

Reopen `selection.json` here to refine presentation or deliberately reanalyse with current code.
Use `analyse_results.py verify REPORT` to check saved outputs, `--verify-sources` to also check original evidence,
and `analyse_results.py rebuild REPORT --output NEW_DIRECTORY` to require unchanged analysis code/environment
and identical scientific CSV/LaTeX. Neither software check qualifies hardware measurements.


In [ ]:
SAVE = os.environ.get('PQC_ANALYSIS_PUBLISH') == '1'
NAME = 'thesis-analysis'
OUTPUT = os.environ.get('PQC_ANALYSIS_OUTPUT')
if SAVE and spec is not None:
    output, report = analysis.publish(spec, name=NAME, output=OUTPUT)
    print('Saved:', output)
    for name in ('tables.html', 'selection.json', 'metrics.csv', 'observations.csv', 'methods.json'):
        display(FileLink(os.path.relpath(output / name, Path.cwd())))
elif not SAVE:
    print('Preview only. Set SAVE = True to save a revision.')
